In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cài đặt các gói cần thiết

In [ ]:
!pip install -U sentence-transformers datasets pandas torch
!pip install --upgrade torch torchvision torchaudio

## Khai báo và chuẩn bị hàm tải dữ liệu

### Import thư viện và cấu hình cơ bản

In [ ]:
%cd /content/drive/MyDrive/Project_ML

/content/drive/MyDrive/Project_ML


In [ ]:
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader
import random
import torch
from datasets import load_dataset

# Cấu hình tham số
MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 16
EPOCHS = 3
OUTPUT_PATH = "./vietlaw-bge-m3-finetuned"

print(f"Thiết bị đang sử dụng: {'GPU (cuda)' if torch.cuda.is_available() else 'CPU'}")

Thiết bị đang sử dụng: GPU (cuda)


### Hàm chuẩn hóa cấu trúc dữ liệu

In [ ]:
def load_adamwhite625():
    print("Đang tải dataset: adamwhite625/vietnam-legal-qa...")
    try:
        ds = load_dataset("adamwhite625/vietnam-legal-qa", split="train")
        standardized_data = []
        for row in ds:
            q = str(row.get('question', '')).strip()
            a = str(row.get('law_content', '')).strip()
            if q and a and len(a) > 50:
                standardized_data.append({"query": q, "pos": a})
        print(f"-> Thu thập được {len(standardized_data)} cặp.")
        return standardized_data
    except Exception as e:
        print(f"Lỗi: {e}")
        return []

def load_huyydangg():
    print("Đang tải dataset: huyydangg/LEGAL-EVAL-Dataset...")
    try:
        ds = load_dataset("huyydangg/LEGAL-EVAL-Dataset", split="test")
        standardized_data = []
        for row in ds:
            q = str(row.get('anchor', '')).strip()
            a = str(row.get('positive', '')).strip()
            if q and a and len(a) > 50:
                standardized_data.append({"query": q, "pos": a})
        print(f"-> Thu thập được {len(standardized_data)} cặp.")
        return standardized_data
    except Exception as e:
        print(f"Lỗi: {e}")
        return []

def load_thangvip():
    print("Đang tải dataset: thangvip/vietnamese-legal-qa...")
    try:
        ds = load_dataset("thangvip/vietnamese-legal-qa", split="train")
        standardized_data = []
        for row in ds:
            qa_pairs = row.get('generated_qa_pairs', [])

            if isinstance(qa_pairs, list):
                for pair in qa_pairs:
                    # Truy cập vào 'question' và 'answer' trong mỗi dict
                    q = str(pair.get('question', '')).strip()
                    a = str(pair.get('answer', '')).strip()
                    if q and a and len(a) > 50:
                        standardized_data.append({"query": q, "pos": a})
        print(f"-> Thu thập được {len(standardized_data)} cặp.")
        return standardized_data
    except Exception as e:
        print(f"Lỗi: {e}")
        return []

## Kéo dữ liệu từ Hugging Face và phân chia

### Tải, trộn và chia tập Train/Test

In [ ]:
data1 = load_adamwhite625()
data2 = load_huyydangg()
data3 = load_thangvip()

# Gộp và trộn ngẫu nhiên
all_data = data1 + data2 + data3
random.shuffle(all_data)
print(f"\nTổng số cặp (Q&A) hợp lệ thu thập được từ 3 nguồn: {len(all_data)}")

# Chia tỷ lệ 90% Train - 10% Test
split_idx = int(len(all_data) * 0.9)
train_data = all_data[:split_idx]
test_data = all_data[split_idx:]

print(f"Số lượng tập Train: {len(train_data)} | Tập Test (Đánh giá): {len(test_data)}")

Đang tải dataset: adamwhite625/vietnam-legal-qa...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


-> Thu thập được 4837 cặp.
Đang tải dataset: huyydangg/LEGAL-EVAL-Dataset...


-> Thu thập được 3873 cặp.
Đang tải dataset: thangvip/vietnamese-legal-qa...
-> Thu thập được 29140 cặp.

Tổng số cặp (Q&A) hợp lệ thu thập được từ 3 nguồn: 37850
Số lượng tập Train: 34065 | Tập Test (Đánh giá): 3785


## Thiết lập Mô hình và Công cụ Đánh giá

### Load mô hình và DataLoader

In [ ]:
print("Đang tải mô hình BGE-M3 gốc (Sẽ mất chút thời gian)...")
model = SentenceTransformer(MODEL_NAME)

model.max_seq_length = 768

# Đưa dữ liệu vào định dạng InputExample của SentenceTransformers
train_examples = [InputExample(texts=[d["query"], d["pos"]]) for d in train_data]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

# Sử dụng MultipleNegativesRankingLoss
train_loss = losses.MultipleNegativesRankingLoss(model)
print("Setup DataLoader và Loss function hoàn tất!")

Đang tải mô hình BGE-M3 gốc (Sẽ mất chút thời gian)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Setup DataLoader và Loss function hoàn tất!


### Cấu hình Evaluator (Bộ chấm điểm RAG)

In [ ]:
# Chuyển tập Test thành dạng từ điển {id: text} để mô phỏng Database FAISS
queries = {str(i): d["query"] for i, d in enumerate(test_data)}
corpus = {str(i): d["pos"] for i, d in enumerate(test_data)}
relevant_docs = {str(i): {str(i)} for i in range(len(test_data))}

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legal-eval",
    show_progress_bar=True
)
print("Setup Evaluator hoàn tất!")

Setup Evaluator hoàn tất!


## Chạy Đánh giá và Huấn luyện (Fine-tuning)

### Baseline

In [ ]:
print("="*50)
print("1. ĐÁNH GIÁ MÔ HÌNH GỐC (Chưa Fine-tune)")
print("="*50)
base_results = evaluator(model, output_path="./eval_base")
print("-> Đã lưu file kết quả tại thư mục ./eval_base")

1. ĐÁNH GIÁ MÔ HÌNH GỐC (Chưa Fine-tune)


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [01:13<00:00, 73.60s/it]


-> Đã lưu file kết quả tại thư mục ./eval_base


In [ ]:
import torch, gc

# Dọn rác hệ thống và xả bộ nhớ GPU
gc.collect()
torch.cuda.empty_cache()
print(f"Bộ nhớ GPU đã dọn dẹp. Đang trống: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

Bộ nhớ GPU đã dọn dẹp. Đang trống: 2396.00 MB


### FINE-TUNING

In [ ]:
print("="*50)
print("2. BẮT ĐẦU FINE-TUNING...")
print("="*50)
# model[0].auto_model.config.use_cache = False
# # BẬT GRADIENT CHECKPOINTING (Cứu RAM)
# model[0].auto_model.gradient_checkpointing_enable()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=100,
    evaluator=evaluator,
    evaluation_steps=500, # Cứ 500 bước cập nhật điểm số 1 lần
    use_amp=True,
    output_path=OUTPUT_PATH,
    show_progress_bar=True
)
print(f"\n✅ Hoàn tất Fine-tuning! Trọng số mới đã được lưu tại: {OUTPUT_PATH}")

2. BẮT ĐẦU FINE-TUNING...


Step,Training Loss,Validation Loss,Legal-eval Cosine Accuracy@1,Legal-eval Cosine Accuracy@3,Legal-eval Cosine Accuracy@5,Legal-eval Cosine Accuracy@10,Legal-eval Cosine Precision@1,Legal-eval Cosine Precision@3,Legal-eval Cosine Precision@5,Legal-eval Cosine Precision@10,Legal-eval Cosine Recall@1,Legal-eval Cosine Recall@3,Legal-eval Cosine Recall@5,Legal-eval Cosine Recall@10,Legal-eval Cosine Ndcg@10,Legal-eval Cosine Mrr@10,Legal-eval Cosine Map@100
500,0.012529,No log,0.939234,0.978600,0.988375,0.995509,0.939234,0.326200,0.197675,0.099551,0.939234,0.978600,0.988375,0.995509,0.969163,0.960508,0.960735
1000,0.004560,No log,0.930515,0.976222,0.987847,0.994452,0.930515,0.325407,0.197569,0.099445,0.930515,0.976222,0.987847,0.994452,0.965129,0.955435,0.955707
1500,0.003904,No log,0.931044,0.973844,0.982563,0.992074,0.931044,0.324615,0.196513,0.099207,0.931044,0.973844,0.982563,0.992074,0.963166,0.953720,0.954178
2000,0.003765,No log,0.933157,0.977543,0.988639,0.994716,0.933157,0.325848,0.197728,0.099472,0.933157,0.977543,0.988639,0.994716,0.966413,0.957051,0.957421
2130,0.003765,No log,0.933950,0.980185,0.991017,0.997094,0.933950,0.326728,0.198203,0.099709,0.933950,0.980185,0.991017,0.997094,0.968077,0.958464,0.958626
2500,0.004080,No log,0.936856,0.981506,0.990225,0.996037,0.936856,0.327169,0.198045,0.099604,0.936856,0.981506,0.990225,0.996037,0.969107,0.960163,0.960378
3000,0.002826,No log,0.938441,0.981770,0.989432,0.996037,0.938441,0.327257,0.197886,0.099604,0.938441,0.981770,0.989432,0.996037,0.969996,0.961355,0.961576
3500,0.000877,No log,0.939498,0.980449,0.988904,0.995773,0.939498,0.326816,0.197781,0.099577,0.939498,0.980449,0.988904,0.995773,0.969891,0.961349,0.961591
4000,0.002793,No log,0.936328,0.979921,0.988639,0.994980,0.936328,0.326640,0.197728,0.099498,0.936328,0.979921,0.988639,0.994980,0.967983,0.959041,0.959369
4260,0.002793,No log,0.942140,0.980978,0.988904,0.995773,0.942140,0.326993,0.197781,0.099577,0.942140,0.980978,0.988904,0.995773,0.971216,0.963096,0.963351


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:20<00:00, 20.43s/it]



✅ Hoàn tất Fine-tuning! Trọng số mới đã được lưu tại: ./vietlaw-bge-m3-finetuned


In [ ]:
import torch, gc

# Dọn rác hệ thống và xả bộ nhớ GPU
gc.collect()
torch.cuda.empty_cache()
print(f"Bộ nhớ GPU đã dọn dẹp. Đang trống: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

Bộ nhớ GPU đã dọn dẹp. Đang trống: 22298.00 MB


### Evaluate

In [ ]:
print("="*50)
print("3. ĐÁNH GIÁ MÔ HÌNH SAU KHI FINE-TUNE")
print("="*50)

# Lúc này biến `model` trong bộ nhớ đã mang trọng số mới
finetuned_results = evaluator(model, output_path="./eval_finetuned")
print("-> Đã lưu file kết quả tại thư mục ./eval_finetuned")

3. ĐÁNH GIÁ MÔ HÌNH SAU KHI FINE-TUNE


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:20<00:00, 20.13s/it]


-> Đã lưu file kết quả tại thư mục ./eval_finetuned


In [ ]:
!zip -r vietlaw-bge-m3-finetuned.zip vietlaw-bge-m3-finetuned/

  adding: vietlaw-bge-m3-finetuned/ (stored 0%)
  adding: vietlaw-bge-m3-finetuned/eval/ (stored 0%)
  adding: vietlaw-bge-m3-finetuned/eval/Information-Retrieval_evaluation_legal-eval_results.csv (deflated 65%)
  adding: vietlaw-bge-m3-finetuned/tokenizer.json (deflated 77%)
  adding: vietlaw-bge-m3-finetuned/1_Pooling/ (stored 0%)
  adding: vietlaw-bge-m3-finetuned/1_Pooling/config.json (deflated 59%)
  adding: vietlaw-bge-m3-finetuned/2_Normalize/ (stored 0%)
  adding: vietlaw-bge-m3-finetuned/config_sentence_transformers.json (deflated 41%)
  adding: vietlaw-bge-m3-finetuned/config.json (deflated 51%)
  adding: vietlaw-bge-m3-finetuned/model.safetensors (deflated 21%)
  adding: vietlaw-bge-m3-finetuned/sentence_bert_config.json (deflated 9%)
  adding: vietlaw-bge-m3-finetuned/tokenizer_config.json (deflated 49%)
  adding: vietlaw-bge-m3-finetuned/README.md (deflated 74%)
  adding: vietlaw-bge-m3-finetuned/modules.json (deflated 62%)
